# Graphing Runnables

In [ ]:
# run the line of code below to check the version of langchain in the current environment.
# substitute "langchain" with any other package name to check their version.

In [ ]:
pip show langchain

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [ ]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 100)

In [ ]:
string_parser = StrOutputParser()

In [ ]:
# Use of Passthrough
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [ ]:
chain_long.get_graph().print_ascii()

# RunnableParallel

In [ ]:
from langchain.schema.runnable import RunnableParallel

In [ ]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

In [ ]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [ ]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [ ]:
chain_parallel.invoke({'programming language':'Python'})

In [ ]:
chain_parallel.get_graph().print_ascii()

In [ ]:
%%time
chain_books.invoke({'programming language':'Python'})

In [ ]:
%%time
chain_projects.invoke({'programming language':'Python'})

In [ ]:
%%time
# keep in mind the output is dictionary
chain_parallel.invoke({'programming language':'Python'})

In [ ]:
# if you run the above three, you will reach to the conclusion that running in parallel is more time efficient

# Piping a RunnableParallel with Other Runnables

In [ ]:
# add a new variable. Expecting the output from books and projects. We require the completion of expected time.
chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [ ]:
# increase the tokens to 500
chat = ChatOpenAI(model_name = 'gpt-4', 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 500)

In [ ]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [ ]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [ ]:
chain_parallel.invoke({'programming language':'Python'})

In [ ]:
# construct a chain and feed this as an input as the chain_time variable

In [ ]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [ ]:
# removed the RunnableParallel wrapper because chat_template_time (other Runnable) will automatically take care of it. 
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [ ]:
print(chain_time2.invoke({'programming language':'Python'}))

In [ ]:
chain_time2.get_graph().print_ascii()

# RunnableLambda

In [ ]:
# RunnableLambda lets you wrap any Python function and use it as a step inside an LCEL chain.
# That means you can take any Python logic — simple or complex — and plug it directly into your LLM pipeline.
# RunnableLambda is used whenever you need to do something that isn’t an LLM call but still belongs inside your chain.
# RunnableLambda is how you inject custom Python logic into an LCEL chain — preprocessing, postprocessing, routing, or 
# integrating external tools — making your pipelines flexible and production‑ready
from langchain_core.runnables import RunnableLambda

In [ ]:
find_sum = lambda x: sum(x)

In [ ]:
find_sum([1, 2, 5])

In [ ]:
find_square = lambda x: x**2

In [ ]:
find_square(8)

In [ ]:
runnable_sum = RunnableLambda(lambda x: sum(x))

In [ ]:
runnable_sum.invoke([1, 2, 5])

In [ ]:
runnable_square = RunnableLambda(lambda x: x**2)

In [ ]:
runnable_square.invoke(8)

In [ ]:
chain = runnable_sum | runnable_square

In [ ]:
chain.invoke([1, 2, 5])

In [ ]:
chain.get_graph().print_ascii()

# The @chain Decorator

In [ ]:
# The  decorator runs because it converts your function into a LangChain Runnable, giving 
# it LCEL execution behavior instead of normal Python function behavior.
from langchain_core.runnables import chain

In [ ]:
def find_sum(x):
    return sum(x)

def find_square(x):
    return x**2

In [ ]:
chain1 = RunnableLambda(find_sum) | RunnableLambda(find_square)

In [ ]:
chain1.invoke([1, 2, 5])

In [ ]:
@chain
def runnable_sum(x):
    return sum(x)

@chain
def runnable_square(x):
    return x**2

In [ ]:
type(runnable_sum), type(runnable_square)

In [ ]:
chain2 = runnable_sum | runnable_square

In [ ]:
chain2.invoke([1, 2, 5])